In [2]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [16]:
best_feat = 'ibi_median'
dataFiles = glob.glob('data/data_to_predict/dados_analise/*.zip')
output_folder = './results/predict'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

['data/data_to_predict/dados_analise/2026-04-16 17.41.32.zip', 'data/data_to_predict/dados_analise/2026-04-23 17.58.52.zip', 'data/data_to_predict/dados_analise/2026-04-14 17.19.07.zip', 'data/data_to_predict/dados_analise/2026-04-24 17.31.38.zip', 'data/data_to_predict/dados_analise/2026-04-22 10.43.23.zip', 'data/data_to_predict/dados_analise/2026-04-24 19.33.09.zip', 'data/data_to_predict/dados_analise/2026-04-17 19.18.35.zip']


In [10]:
root_folder = "./data/data_to_predict/unwrapped_data"    
for file in dataFiles:
    print(file)
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)

    a = chr.intellicage_unwrapper([file], root_folder=os.path.join(root_folder, sub_folder), sampling_interval = '30T')
   
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)

data/data_to_predict/dados_analise/2026-04-16 17.41.32.zip
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_1.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_10.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_11.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_12.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_2.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_3.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_4.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_5.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_6.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_7.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/animal_8.txt
File saved in ./data/data_to_predict/unwrapped_data/2026-04-16_17/ani

In [13]:
def get_intellicage_start_end_date(file):
    file = open(file, 'r')
    lines = file.readlines()
    lines = [line.strip() for line in lines]
    file.close()

    date_type = "%Y-%m-%d %H:%M:%S"        
    start_date = datetime.strptime(lines[1].split("Start date: ", 1)[1].strip(), date_type)
    end_date = datetime.strptime(lines[2].split("End date: ", 1)[1].strip(), date_type)
    #print(f"Start date {start_date} end date {end_date}")
    
    # Calculate number of days
    num_days = (end_date.date() - start_date.date()).days + 1
    
    return num_days

def read_data(file, labels_dict, name="", zt_0_time = None, type = 'intellicage'):

    protocol = None

    try:
        #print(f"Reading file {file}") 
        num_days = get_intellicage_start_end_date(file)
        #print(f"Number of days: {num_days}")
        labels_dict['cycle_days'] = [num_days+1]
        protocol = chr.read_protocol(name, file, zt_0_time = zt_0_time, labels_dict = labels_dict, type = type, consider_first_day = True)
    except Exception as e:
        print(f"Error reading file {file}: {e}")

    return protocol

# Split animal_3 protocol into per-day dataframes
def split_animal_protocol_by_day(animals_protocols, animal):
    animal_df = animals_protocols[animal].data.copy()
    animal_df['date'] = animal_df.index.normalize()

    # Build a dict of day -> dataframe
    animal_by_day = {
        d: animal_df[animal_df['date'] == d].drop(columns='date')
        for d in sorted(animal_df['date'].unique())
    }
    ks = list(animal_by_day.keys())
    print(f"Animal {animal} Days found:", len(ks))
    return animal_by_day


def build_animal_protocols2(
        animals_files, 
        zt_0_time=20, 
        labels_dict={'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}):

 
    animals_protocols = {}
    reads = []


    for k, v in animals_files.items():
        print(f"Animal2 {k}")
        if k not in reads:

            
                protocol = read_data(v[0], zt_0_time=zt_0_time, labels_dict=labels_dict)

                for i in range(1, len(v)):
                    try:
                        protocol.concat_protocols(read_data(v[i], zt_0_time=zt_0_time, labels_dict=labels_dict), method='sum')
                    except Exception as e:
                        print(f"Error trying to concatenate {v} with {v[i]}")


         
                protocol.resample('15T', method='sum')
                protocol.apply_filter(type = 'savgol')
                animals_protocols[f"animal_{k}"] = protocol
                reads.append(k)
 
        else:
            print(f"Animal {k} already read, skipping.")

    
    animals_by_day = {}
    for k in animals_protocols.keys():
        animals_by_day[k] = split_animal_protocol_by_day(animals_protocols, k)

    return animals_protocols, animals_by_day

def get_sorted_animals_files(individual_files, animals=None):

    animals_files = {}

    for file in individual_files:
        # Extract animal number from filename
        # Assuming format like "data_unwrapped_animal_1.txt"
        animal_number = int(file.split("_")[-1].split(".")[0])  # Gets "1" from "animal_1"

        if animals is not None and animal_number not in animals:
            continue
        
        if animal_number not in animals_files:
            animals_files[animal_number] = []
        
        animals_files[animal_number].append(file)

    # Sort files for each animal
    for animal in animals_files:
        animals_files[animal].sort()

    # Sort the dictionary by animal number
    animals_files = dict(sorted(animals_files.items(), key=lambda x: int(x[0])))

    for k,v in animals_files.items():
        print(f"Animal {k}: {len(v)}")

    return animals_files

#usando sum para concatenar os protocolos gera resultado melhor que o last
#Se eu quiser ler os protocolos eu preciso do zt_0_time, neste caso estou usando 20horas.
#Tambem precisariamos definir o labels dict, neste caso o que utilizariamos?
zt_0_time = 20  #Para gerar o actograma é usado o zt dado de quando a luz é acesa, que será as 20 horas. 
labels_dict = {'cycle_types': ['LD'], 'test_labels': ['1_control_dl'], 'cycle_days': [1]}
animals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols2(animals_files, zt_0_time=zt_0_time, labels_dict=labels_dict)


Animal 1: 7
Animal 2: 7
Animal 3: 7
Animal 4: 7
Animal 5: 7
Animal 6: 7
Animal 7: 7
Animal 8: 7
Animal 9: 7
Animal 10: 7
Animal 11: 7
Animal 12: 7
Animal2 1
Resampling data to 15T using method sum
savgol True
Animal2 2
Resampling data to 15T using method sum
savgol True
Animal2 3
Resampling data to 15T using method sum
savgol True
Animal2 4
Resampling data to 15T using method sum
savgol True
Animal2 5
Resampling data to 15T using method sum
savgol True
Animal2 6
Resampling data to 15T using method sum
savgol True
Animal2 7
Resampling data to 15T using method sum
savgol True
Animal2 8
Resampling data to 15T using method sum
savgol True
Animal2 9
Resampling data to 15T using method sum
savgol True
Animal2 10
Resampling data to 15T using method sum
savgol True
Animal2 11
Resampling data to 15T using method sum
savgol True
Animal2 12
Resampling data to 15T using method sum
savgol True
Animal animal_1 Days found: 15
Animal animal_2 Days found: 15
Animal animal_3 Days found: 15
Animal animal

In [20]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)


Saving features on ./results/predict/basic_features.csv
Saving temporal features on ./results/predict/temporal_features.csv
Saving all features on ./results/predict/all_features.csv


In [30]:
feature_rhos, proxies_raw = build_all_proxies(
    all_features, feature_cols, X_scaled,
    k=3, best_feat_idx=best_feat
)
scores = proxies_raw['Best feature (ibi_median)']
pred_rank = rankdata(scores, method='ordinal')
print(pred_rank)


[ 1  2  3  4  5  6  7  8  9 10 12 11]
